# Merge Conflict Prediction with MLPClassifier

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, classification_report, confusion_matrix, make_scorer
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample

model_path = "mlp_merge_conflict_model.pkl"

## 📊 Data Preprocessing

We start by loading the dataset and performing the following steps:

- **Dropping non-predictive identifiers**: `commit`, `parent1`, `parent2`, `ancestor`
- **Filling missing values**: With median for numeric columns
- **Train/Test split**: Stratified to preserve class ratio
- **Manual oversampling**: To balance the training dataset
- **Feature scaling**: Standardized with `StandardScaler`


In [2]:
df = pd.read_csv('MergeConflictsDataset.csv', sep=';')

# Drop non-predictive ID columns
df.drop(columns=['commit', 'parent1', 'parent2', 'ancestor'], inplace=True)

# Handle missing values
if df.isnull().values.any():
    df = df.fillna(df.median(numeric_only=True))

df.head()

,is pr,added lines,deleted lines,devs parent1,devs parent2,time,nr files,added files,deleted files,renamed files,...,add,remove,use,delete,change,messages_min,messages_max,messages_mean,messages_median,conflict
0,1,5,0,0,1,23,0,0,0,0,...,0,0,0,0,0,20,65,35.40000,20.0,0
1,0,1166,11267,1,2,371,3,7,199,2,...,0,0,0,0,0,31,117,58.56383,53.5,1
2,1,0,0,0,1,22,0,0,0,0,...,0,0,0,0,0,18,18,18.00000,18.0,0
3,1,0,0,2,1,24,1,0,0,0,...,0,0,0,0,0,22,63,38.80000,31.0,0
4,0,0,0,1,2,2,1,0,0,0,...,0,0,0,0,0,31,56,43.50000,43.5,1


In [3]:
# Split into features and target
X = df.drop(columns=['conflict'])
y = df['conflict']

# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

🔁 Manual Oversampling + Scaling

- The training set will be **balanced** by upsampling the minority class (`conflict = 1`).
- Oversampling has to be applied **only to the training data** to prevent data leakage.
- Feature scaling using `StandardScaler` ensures better convergence for the MLP model, which is sensitive to feature magnitudes.

In [4]:
# Concatenate training features and labels
train_df = pd.concat([X_train, y_train], axis=1)

# Split into majority and minority classes
majority = train_df[train_df.conflict == 0]
minority = train_df[train_df.conflict == 1]

# Upsample the minority class
minority_upsampled = resample(minority, 
                               replace=True,
                               n_samples=len(majority),
                               random_state=42)

# Combine majority class with upsampled minority class
train_balanced = pd.concat([majority, minority_upsampled])

# Separate X and y
X_train_bal = train_balanced.drop(columns=['conflict'])
y_train_bal = train_balanced['conflict']

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

### 🧠 Defining and Tuning the MLP Classifier

In this step, we configure a **Multi-Layer Perceptron (MLP) Classifier**, which is a type of feedforward neural network suitable for binary classification tasks like ours.

- We use `early_stopping=True` to **prevent overfitting** by halting training when validation performance no longer improves.
- A **grid search** is performed to find the best combination of hyperparameters:
  - `hidden_layer_sizes`: Controls the architecture (number of neurons and layers).
  - `alpha`: Regularization term to penalize large weights (helps prevent overfitting).
  - `learning_rate_init`: Starting learning rate for the optimizer.

The grid search is evaluated using **recall** as the scoring metric, emphasizing the correct identification of conflict cases (minority class), which is crucial in imbalanced datasets.


In [5]:
mlp = MLPClassifier(max_iter=500, early_stopping=True, random_state=42)

# Define hyperparameter grid
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.01]
}

# Grid search (maximize recall)
grid_search = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid,
    scoring=make_scorer(recall_score),
    cv=3,
    verbose=1,
    n_jobs=-1
)

### 💾 Load or Train the MLPClassifier

To improve efficiency and avoid retraining the model every time, we implement logic to:

- **Check if a trained model already exists** on disk (`mlp_merge_conflict_model.pkl`)
- **Load the model** if it exists, ensuring reproducibility and faster iteration
- **Train a new model using GridSearchCV** if no saved model is found
- **Save the trained model** to disk using `pickle` for future use

This approach enables faster experimentation and makes the workflow more robust and scalable, especially in environments where compute resources are limited or training time is significant.

In [6]:
if os.path.exists(model_path):
    print("🔁 Loading model from disk...")
    with open(model_path, 'rb') as f:
        mlp_best = pickle.load(f)
else:
    print("🚀 Training model via GridSearchCV...")
    grid_search.fit(X_train_scaled, y_train_bal)
    
    mlp_best = grid_search.best_estimator_

    with open(model_path, 'wb') as f:
        pickle.dump(mlp_best, f)
    print("✅ Model saved to:", model_path)

🔁 Loading model from disk...


### 🧪 Model Evaluation on Test Set

With the model trained (or loaded), we now evaluate its performance on the **unseen test set**. 

Our objective, based on the assignment requirement (first name starting with "A–K"), is to **maximize Recall** — especially for the minority class (`conflict = 1`), which represents the true conflict cases.

We compute the following evaluation metrics:

- **Recall**: The most critical metric for our task — how many actual conflict cases we correctly identified.
- **Confusion Matrix**: Gives us insight into true positives, false positives, and false negatives.
- **Classification Report**: Summarizes precision, recall, and F1-score for both classes.


In [7]:
y_pred = mlp_best.predict(X_test_scaled)

print("🎯 MLPClassifier with Manual Oversampling")
print("Recall:", recall_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

🎯 MLPClassifier with Manual Oversampling
Recall: 0.8571428571428571
[[4906  195]
 [  42  252]]
              precision    recall  f1-score   support

           0       0.99      0.96      0.98      5101
           1       0.56      0.86      0.68       294

    accuracy                           0.96      5395
   macro avg       0.78      0.91      0.83      5395
weighted avg       0.97      0.96      0.96      5395



### ✅ Evaluation Summary

The model achieved the following results:

- **Overall Recall**: 0.857
- **Recall for Class 1 (conflict)**: 86%  
- **Confusion Matrix**:
  - True Positives (correctly identified conflicts): 252
  - False Negatives (missed conflicts): 42

This performance **strongly aligns with our main goal of maximizing Recall**. We ensured that the model is not underfitting or overfitting and is generalizing well to new, unseen examples.

🎯 **Conclusion**: The MLPClassifier successfully identifies the majority of conflict cases, fulfilling the task requirements.
